In [12]:
import pandas as pd, numpy as np, json, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import defaultdict, Counter

CSV_PATH = "res/e38_18.csv"  # <-- állítsd be

# --- 0) oszlopfelismerés (rugalmas) ---
def guess(cols, keys):
    keys = [k.lower() for k in keys]
    for c in cols:
        lc = c.lower()
        if any(k in lc for k in keys):
            return c
    return None

df = pd.read_csv(CSV_PATH)

em_col   = guess(df.columns, ["exact_match","is_correct","em","correct"])
pat_col  = guess(df.columns, ["calc_pattern","pattern","calc"])
q_col    = guess(df.columns, ["question","prompt","query"])
pnum_col = guess(df.columns, ["pred","model","answer_num","pred_number","pred_value","predicted_value"])
pscl_col = guess(df.columns, ["pred_scale","model_scale","answer_scale","predicted_scale"])
gnum_col = guess(df.columns, ["gold","label","target_num","gold_number","true_value"])
gscl_col = guess(df.columns, ["gold_scale","label_scale","target_scale","true_scale"])

assert em_col and pat_col and q_col, "Hiányzó kulcsoszlop(ok): exact_match / calc_pattern / question"

# --- 1) normalizálás ---
SCALE = {"":1, "thousand":1e3, "million":1e6, "billion":1e9, "percent":1, "%":1}
def to_base(num, scale):
    if pd.isna(num): return np.nan
    s = str(scale).strip().lower() if pd.notna(scale) else ""
    return float(num) * SCALE.get(s, 1)

def norm_series(num_col, scl_col):
    if num_col is None: return pd.Series(np.nan, index=df.index)
    nums = pd.to_numeric(df[num_col], errors="coerce")
    scls = df[scl_col] if scl_col in df.columns else ""
    return [to_base(n, s) for n, s in zip(nums, scls)]

df["_em"] = df[em_col].astype(str).str.lower().isin(["true","1","yes","t"])
df["_pred_base"] = norm_series(pnum_col, pscl_col)
df["_gold_base"] = norm_series(gnum_col, gscl_col)

# --- 2) hibacímkézés ---
def classify(row, eps=1e-8):
    if row["_em"]: return "ok"
    pb, gb = row["_pred_base"], row["_gold_base"]
    q = str(row[q_col]).lower()
    lab = []

    if np.isfinite(pb) and np.isfinite(gb):
        if abs(pb + gb) <= max(1e-8, 1e-6*max(1,abs(gb))):
            lab.append("sign_flip")
        if gb != 0:
            ratio = pb/gb
            for k in [0.001,0.01,0.1,10,100,1000]:
                if np.isfinite(ratio) and abs(ratio - k) <= 0.02:
                    lab.append(f"scale_mismatch_{k}x")
    if re.search(r"(percentage change|yoy|growth)", q):
        lab.append("pct_vs_pp_candidate")
    if ("average" in q or "avg" in q) and re.search(r"(year|previous)", q):
        lab.append("avg_window_candidate")
    if re.search(r"(largest|max(imum)?|highest|smallest|min(imum)?|lowest)", q):
        lab.append("extremum_candidate")

    return "+".join(lab) if lab else "other"

df["error_type"] = df.apply(classify, axis=1)

# --- 3) TF-IDF klaszterezés 'other' hibákra calc_pattern-ön belül ---
def find_kmeans_clusters(texts, min_k=2, max_k=6):
    if len(texts) < 15:
        return None, None, None
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2)
    X = vec.fit_transform(texts)
    best_k, best_score, best_model = None, -1, None
    for k in range(min_k, min(max_k, len(texts)-1)+1):
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labs = km.fit_predict(X)
        if len(set(labs)) > 1:
            sc = silhouette_score(X, labs)
            if sc > best_score:
                best_k, best_score, best_model = k, sc, km
    return best_model, vec, best_score

clusters = []
for cp, g in df[(~df["_em"])].groupby(pat_col):
    # bonts error_type szerint
    for et, gg in g.groupby("error_type"):
        if et != "other":
            clusters.append({"calc_pattern":cp,"error_type":et,"cluster_id":-1,"n":len(gg),
                             "silhouette":None,"top_terms":[]})
        else:
            model, vec, sc = find_kmeans_clusters(gg[q_col].astype(str).tolist())
            if model is None:
                clusters.append({"calc_pattern":cp,"error_type":"other","cluster_id":-1,"n":len(gg),
                                 "silhouette":None,"top_terms":[]})
            else:
                X = vec.transform(gg[q_col].astype(str).tolist())
                labs = model.predict(X)
                terms = np.array(vec.get_feature_names_out())
                for cid in sorted(set(labs)):
                    mask = (labs==cid)
                    n = int(mask.sum())
                    centroid = model.cluster_centers_[cid]
                    # top 8 ngram
                    top_idx = centroid.argsort()[-8:][::-1]
                    top_terms = terms[top_idx].tolist()
                    clusters.append({"calc_pattern":cp,"error_type":"tfidf_cluster",
                                     "cluster_id":int(cid),"n":n,"silhouette":float(sc),
                                     "top_terms":top_terms})

cldf = pd.DataFrame(clusters).sort_values(["calc_pattern","error_type","cluster_id","n"], ascending=[True,True,True,False])
cldf.to_csv("cluster_summary.csv", index=False)

# --- 4) szabály-javaslatok ---
RULES = {
  "sign_flip": "In financial tables, treat numbers in parentheses as negative values.",
  "scale_mismatch": "Normalize units before computing; return (number, scale) consistently with {'', 'thousand','million','billion','percent'}.",
  "pct_vs_pp_candidate": "‘percentage change’ = (new-old)/old*100; ‘change in percentage’ = new% - old% (percentage points).",
  "avg_window_candidate": "Year average = (value(y) + value(y-1)) / 2.",
  "extremum_candidate": "For ‘largest/smallest’, select the max/min after unit normalization; do not sum or average.",
  "header_guard": "Never index columns by position; select columns by header text (case-insensitive, stripped)."
}

patches = defaultdict(lambda: {"instructions": Counter(), "evidence": []})

def add_patch(key, instr, evidence):
    patches[key]["instructions"][instr] += 1
    if len(patches[key]["evidence"]) < 5:
        patches[key]["evidence"].append(evidence)

# szabályok az ismert error_type-okra
for _, row in df[(~df["_em"])].iterrows():
    cp, et = row[pat_col], row["error_type"]
    if "sign_flip" in et:
        add_patch((cp,"sign_flip"), RULES["sign_flip"], row[q_col])
    if "scale_mismatch" in et:
        add_patch((cp,"scale_mismatch"), RULES["scale_mismatch"], row[q_col])
    if "pct_vs_pp_candidate" in et:
        add_patch((cp,"pct_vs_pp_candidate"), RULES["pct_vs_pp_candidate"], row[q_col])
    if "avg_window_candidate" in et:
        add_patch((cp,"avg_window_candidate"), RULES["avg_window_candidate"], row[q_col])
    if "extremum_candidate" in et:
        add_patch((cp,"extremum_candidate"), RULES["extremum_candidate"], row[q_col])

# TF-IDF klaszterek kulcsszavak alapján (heur)
KEYMAP = [
    (r"(percentage change|yoy|growth)", RULES["pct_vs_pp_candidate"]),
    (r"(average|avg).*(year|previous)", RULES["avg_window_candidate"]),
    (r"(largest|max|highest|smallest|min|lowest)", RULES["extremum_candidate"]),
]
for r in clusters:
    if r["error_type"]!="tfidf_cluster": continue
    text = " ".join(r["top_terms"])
    assigned = False
    for pat, rule in KEYMAP:
        if re.search(pat, text):
            add_patch((r["calc_pattern"],"tfidf_cluster_"+str(r["cluster_id"])), rule, text)
            assigned = True
    if not assigned:
        add_patch((r["calc_pattern"],"tfidf_cluster_"+str(r["cluster_id"])),
                  "Clarify units/time window; ensure text-based column selection.", text)

# mindig javasoljunk egy globális header_guard-ot is, mert CGA-nál ez tipikus
for cp in df[pat_col].unique():
    add_patch((cp,"header_guard"), RULES["header_guard"], "global")

# export
export = []
for (cp, et), data in patches.items():
    instr, _ = data["instructions"].most_common(1)[0]
    export.append({
        "calc_pattern": cp,
        "error_type": et,
        "suggested_instruction": instr,
        "support": int(sum(data["instructions"].values())),
        "evidence_examples": data["evidence"]
    })

with open("prompt_patches.json","w",encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

print("OK → cluster_summary.csv + prompt_patches.json")


OK → cluster_summary.csv + prompt_patches.json


In [24]:
[f for f in pd.DataFrame(export)["suggested_instruction"].unique()]

['‘percentage change’ = (new-old)/old*100; ‘change in percentage’ = new% - old% (percentage points).',
 'Year average = (value(y) + value(y-1)) / 2.',
 'For ‘largest/smallest’, select the max/min after unit normalization; do not sum or average.',
 'Clarify units/time window; ensure text-based column selection.',
 'Never index columns by position; select columns by header text (case-insensitive, stripped).']